# LeetCode 3692 — Majority Frequency Characters bằng Python

Link đề: https://leetcode.com/problems/majority-frequency-characters/ · Độ khó: **Easy** · Chủ đề: Hash Table, String, Counting

Skeleton LeetCode:

```python
class Solution:
    def majorityFrequencyGroup(self, s: str) -> str:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: **nhóm** các ký tự theo **tần suất** (các ký tự xuất hiện cùng số lần vào một nhóm); chọn nhóm có **nhiều ký tự khác nhau nhất**; nếu hòa thì chọn nhóm có **tần suất lớn hơn**; trả về các ký tự của nhóm đó (thứ tự tùy ý).
2. Dùng "đếm hai tầng": đếm tần suất ký tự (`Counter`), rồi **gom** ký tự theo tần suất (từ điển `tần suất -> danh sách ký tự`).
3. Chọn phần tử lớn nhất theo **khóa ghép** `(kích thước nhóm, tần suất)` bằng `max(..., key=...)`.
4. Viết 4 cách: vét cạn theo từng tần suất `1..n`, `Counter` + `defaultdict`, xô theo tần suất (bucket), và `Counter` của các tần suất; đối chiếu bằng test ngẫu nhiên có `random.seed`.
5. Tránh các lỗi: hòa thì chọn tần suất **thấp**, chỉ trả **một** ký tự phổ biến nhất, chọn nhóm theo **tần suất lớn nhất** thay vì kích thước nhóm, và đo nhóm bằng **tổng số lần xuất hiện**.

## 2. Đề bài & ví dụ

Cho chuỗi `s` gồm chữ cái thường. **Nhóm tần suất** là tập các ký tự có **cùng số lần xuất hiện** trong `s`. **Nhóm đa số** (majority frequency group) là nhóm có **nhiều ký tự khác nhau nhất**. Nếu có nhiều nhóm cùng kích thước lớn nhất, chọn nhóm có **tần suất lớn hơn**. Trả về chuỗi gồm các ký tự của nhóm đa số, theo **thứ tự bất kỳ**.

| Ví dụ | s | Output | Giải thích |
|---|---|---|---|
| 1 | `"aaabbbccdddde"` | `"ab"` | `a, b` đều `3` lần (nhóm kích thước 2); `c` `2` lần, `d` `4` lần, `e` `1` lần (mỗi nhóm kích thước 1); `"ba"` cũng hợp lệ |
| 2 | `"abcd"` | `"abcd"` | cả bốn ký tự xuất hiện `1` lần, một nhóm kích thước 4 |
| 3 | `"pfpfgi"` | `"fp"` | `p, f` `2` lần; `g, i` `1` lần; hai nhóm cùng kích thước 2, chọn tần suất lớn hơn (`2`) |

Constraints: `1 <= s.length <= 100`; `s` chỉ gồm chữ cái thường tiếng Anh.

In [1]:
from collections import Counter
s = "aaabbbccdddde"
cnt = Counter(s)
print(cnt)
groups = {}
for ch, f in cnt.items():
    groups.setdefault(f, []).append(ch)
print(groups)

Counter({'d': 4, 'a': 3, 'b': 3, 'c': 2, 'e': 1})
{3: ['a', 'b'], 2: ['c'], 4: ['d'], 1: ['e']}


## 3. Kiến thức nền

### 3.1. `Counter` và tần suất

`Counter(s)` cho `{ký tự: số lần}`. Đây là "tầng một" của bài. Ví dụ `Counter("pfpfgi") = {p: 2, f: 2, g: 1, i: 1}`.

In [2]:
from collections import Counter
cnt = Counter("pfpfgi")
print(cnt)
print(cnt.items())

Counter({'p': 2, 'f': 2, 'g': 1, 'i': 1})
dict_items([('p', 2), ('f', 2), ('g', 1), ('i', 1)])


### 3.2. Gom nhóm theo khóa (`dict` / `defaultdict`)

Muốn ánh xạ `tần suất -> các ký tự`, đảo vai trò khóa/giá trị: `groups[f].append(ch)`. `defaultdict(list)` tự tạo list rỗng khi khóa chưa có; dùng `dict.setdefault(f, [])` cũng được.

In [3]:
from collections import defaultdict
groups = defaultdict(list)
for ch, f in Counter("pfpfgi").items():
    groups[f].append(ch)
print(dict(groups))
print(groups[5])               # khóa chưa có -> list rỗng (và tự tạo khóa)

{2: ['p', 'f'], 1: ['g', 'i']}
[]


### 3.3. `max` với khóa ghép

`max(iterable, key=f)` trả phần tử có `f` lớn nhất. Nếu `f` trả về **tuple** `(a, b)` thì so sánh `a` trước, hòa thì so `b`. Ở đây khóa là `(len(nhóm), tần suất)`: kích thước nhóm lớn nhất, hòa thì tần suất lớn hơn — đúng luật của đề.

In [4]:
groups = {2: ["p", "f"], 1: ["g", "i"]}
best_f = max(groups, key=lambda f: (len(groups[f]), f))
print(best_f, groups[best_f])
print(max([(2, 1), (2, 2), (1, 9)]))

2 ['p', 'f']
(2, 2)


### 3.4. `Counter` của các giá trị (đếm hai tầng)

`Counter(cnt.values())` đếm "bao nhiêu ký tự có tần suất `f`", tức **kích thước từng nhóm** ngay lập tức: `Counter({2: 2, 1: 2})` cho `"pfpfgi"`. Từ đó chọn `f` tốt nhất rồi mới lấy danh sách ký tự.

In [5]:
cnt = Counter("pfpfgi")
size = Counter(cnt.values())
print(size)
print(max(size, key=lambda f: (size[f], f)))

Counter({2: 2, 1: 2})
2


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 1: `s = "aaabbbccdddde"`.

```
tần suất: a=3, b=3, c=2, d=4, e=1
gom nhóm: 3 -> {a, b}   2 -> {c}   4 -> {d}   1 -> {e}
kích thước: 2, 1, 1, 1  -> lớn nhất là nhóm tần suất 3 (kích thước 2)
đáp án: "ab"
```

Ví dụ 3: `"pfpfgi"`: nhóm `2 -> {p, f}`, nhóm `1 -> {g, i}`; cùng kích thước `2` nên ưu tiên **tần suất lớn hơn** (`2`) -> `"fp"`.

Hai bước: **đếm ký tự** rồi **đếm nhóm**; cuối cùng chọn bằng khóa `(kích thước, tần suất)`.

## 5. Cách 1 — Brute Force / baseline

Thử **từng tần suất** `f = 1 .. n`: dựng nhóm `{ch : s.count(ch) == f}`; ghi nhận nhóm có kích thước lớn nhất (hòa thì giữ `f` lớn hơn bằng cách duyệt `f` tăng dần và dùng `>=`).

Nút thắt: `s.count(ch)` duyệt cả chuỗi mỗi lần nên tổng khoảng `O(n * 26 * n)`; với `n <= 100` chấp nhận được nhưng thừa.

In [6]:
class SolutionBrute:
    def majorityFrequencyGroup(self, s):
        best_group, best_f = [], 0
        for f in range(1, len(s) + 1):
            group = [ch for ch in sorted(set(s)) if s.count(ch) == f]
            if len(group) >= len(best_group) and len(group) > 0:
                best_group, best_f = group, f          # dùng >= nên f lớn hơn thắng khi hòa
        return "".join(best_group)

print(SolutionBrute().majorityFrequencyGroup("aaabbbccdddde"))
print(SolutionBrute().majorityFrequencyGroup("abcd"))
print(SolutionBrute().majorityFrequencyGroup("pfpfgi"))

ab
abcd
fp


**Độ phức tạp:** Time `O(n * |Σ| * n)` với `|Σ| <= 26` (mỗi `count` `O(n)`), Space `O(26)`.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** đếm **hai tầng**: (1) tần suất mỗi ký tự; (2) với mỗi tần suất `f`, các ký tự có tần suất `f` tạo thành một nhóm. Số tần suất khác nhau tối đa khoảng `14` (vì `1 + 2 + ... + 14 > 100`), nên chọn nhóm tốt nhất là việc rất nhỏ.

**Quy tắc "nếu… thì…":**

- Nếu nhóm `A` **nhiều ký tự hơn** nhóm `B` -> chọn `A`.
- Nếu hai nhóm **bằng kích thước** -> chọn nhóm có **tần suất lớn hơn**.
- Khóa so sánh: `(len(nhóm), tần suất)` -> `max` theo khóa này.

Invariant: mỗi ký tự xuất hiện trong **đúng một** nhóm; tổng kích thước các nhóm = số ký tự khác nhau của `s`.

In [7]:
def majority(s):
    cnt = Counter(s)
    groups = {}
    for ch, f in cnt.items():
        groups.setdefault(f, []).append(ch)
    print("nhóm:", {f: "".join(sorted(g)) for f, g in groups.items()})
    best = max(groups, key=lambda f: (len(groups[f]), f))
    return "".join(sorted(groups[best]))

print(majority("aaabbbccdddde"))
print(majority("pfpfgi"))

nhóm: {3: 'ab', 2: 'c', 4: 'd', 1: 'e'}
ab
nhóm: {2: 'fp', 1: 'gi'}
fp


## 7. Các cách tiếp theo

Tổng cộng 4 cách: (1) vét cạn theo tần suất; (2) `Counter` + `defaultdict` + `max` khóa ghép; (3) xô theo tần suất (mảng danh sách, duyệt từ tần suất lớn xuống); (4) `Counter` của các tần suất để biết kích thước nhóm trước, rồi lọc ký tự.

### Cách 2 — `Counter` + `defaultdict` + `max` khóa ghép

Đúng như mục 6.

In [8]:
from collections import defaultdict

class SolutionGroup:
    def majorityFrequencyGroup(self, s):
        cnt = Counter(s)
        groups = defaultdict(list)
        for ch, f in cnt.items():
            groups[f].append(ch)
        best = max(groups, key=lambda f: (len(groups[f]), f))
        return "".join(groups[best])

print(sorted(SolutionGroup().majorityFrequencyGroup("aaabbbccdddde")))
print(sorted(SolutionGroup().majorityFrequencyGroup("pfpfgi")))

['a', 'b']
['f', 'p']


Độ phức tạp Cách 2: Time `O(n)` (đếm) + `O(26)` (gom, chọn), Space `O(26)`.

### Cách 3 — Xô theo tần suất (bucket)

Tạo `buckets = [[] for _ in range(n + 1)]`; ký tự có tần suất `f` bỏ vào `buckets[f]`. Duyệt `f` từ **lớn xuống nhỏ** và cập nhật khi nhóm **lớn hơn hẳn** (`>` chặt): vì ta đi từ `f` lớn, nhóm cùng kích thước ở `f` nhỏ hơn sẽ **không** thay thế nhóm đã có -> hòa tự động chọn `f` lớn hơn, không cần khóa ghép.

In [9]:
class SolutionBucket:
    def majorityFrequencyGroup(self, s):
        n = len(s)
        freq = [0] * 26
        for ch in s:
            freq[ord(ch) - 97] += 1
        buckets = [[] for _ in range(n + 1)]
        for i in range(26):
            if freq[i] > 0:
                buckets[freq[i]].append(chr(97 + i))
        best = []
        for f in range(n, 0, -1):               # từ tần suất lớn xuống
            if len(buckets[f]) > len(best):      # > chặt: hòa giữ nhóm có f lớn hơn
                best = buckets[f]
        return "".join(best)

print(SolutionBucket().majorityFrequencyGroup("aaabbbccdddde"))
print(SolutionBucket().majorityFrequencyGroup("abcd"))
print(SolutionBucket().majorityFrequencyGroup("pfpfgi"))

ab
abcd
fp


Độ phức tạp Cách 3: Time `O(n + 26)`, Space `O(n + 26)` (mảng xô độ dài `n + 1`).

### Cách 4 — `Counter` của các tần suất

`size = Counter(cnt.values())` cho kích thước từng nhóm. Chọn `f_best = max(size, key=lambda f: (size[f], f))`. Cuối cùng lọc `[ch for ch, f in cnt.items() if f == f_best]`. Ý tưởng "đếm các đếm" rất hữu ích ở nhiều bài.

In [10]:
class SolutionDoubleCount:
    def majorityFrequencyGroup(self, s):
        cnt = Counter(s)
        size = Counter(cnt.values())
        f_best = max(size, key=lambda f: (size[f], f))
        return "".join(ch for ch, f in cnt.items() if f == f_best)

print(SolutionDoubleCount().majorityFrequencyGroup("aaabbbccdddde"))
print(SolutionDoubleCount().majorityFrequencyGroup("abcd"))
print(SolutionDoubleCount().majorityFrequencyGroup("pfpfgi"))

ab
abcd
pf


Độ phức tạp Cách 4: Time `O(n + 26)`, Space `O(26)`.

## 8. TRACE từng bước

Trace Cách 3 (bucket) với `s = "pfpfgi"` (`n = 6`): `p=2, f=2, g=1, i=1`.

| bước | `f` | `buckets[f]` | `len > len(best)`? | `best` sau |
|---|---|---|---|---|
| 1 | 6 | `[]` | `0 > 0` không | `[]` |
| 2 | 5, 4, 3 | `[]` | không | `[]` |
| 3 | 2 | `[f, p]` | `2 > 0` có | `[f, p]` |
| 4 | 1 | `[g, i]` | `2 > 2` không (hòa, giữ `f` lớn) | `[f, p]` |

Đáp án `"fp"`.

In [11]:
def trace_bucket(s):
    n = len(s)
    freq = [0] * 26
    for ch in s:
        freq[ord(ch) - 97] += 1
    buckets = [[] for _ in range(n + 1)]
    for i in range(26):
        if freq[i]:
            buckets[freq[i]].append(chr(97 + i))
    best = []
    for f in range(n, 0, -1):
        take = len(buckets[f]) > len(best)
        if buckets[f]:
            print(f"f={f}: nhóm {buckets[f]} (kích thước {len(buckets[f])}) so với best {best} -> {'lấy' if take else 'giữ'}")
        if take:
            best = buckets[f]
    print("đáp án =", "".join(best))
    return "".join(best)

trace_bucket("pfpfgi")
trace_bucket("aaabbbccdddde")

f=2: nhóm ['f', 'p'] (kích thước 2) so với best [] -> lấy
f=1: nhóm ['g', 'i'] (kích thước 2) so với best ['f', 'p'] -> giữ
đáp án = fp
f=4: nhóm ['d'] (kích thước 1) so với best [] -> lấy
f=3: nhóm ['a', 'b'] (kích thước 2) so với best ['d'] -> lấy
f=2: nhóm ['c'] (kích thước 1) so với best ['a', 'b'] -> giữ
f=1: nhóm ['e'] (kích thước 1) so với best ['a', 'b'] -> giữ
đáp án = ab


'ab'

## 9. Code nộp LeetCode

In [12]:
from typing import List
from collections import Counter, defaultdict

class Solution:
    def majorityFrequencyGroup(self, s: str) -> str:
        cnt = Counter(s)
        groups = defaultdict(list)
        for ch, f in cnt.items():
            groups[f].append(ch)
        best = max(groups, key=lambda f: (len(groups[f]), f))
        return "".join(groups[best])

print(Solution().majorityFrequencyGroup("aaabbbccdddde"))
print(Solution().majorityFrequencyGroup("abcd"))
print(Solution().majorityFrequencyGroup("pfpfgi"))

ab
abcd
pf


## 10. Vì sao thuật toán đúng

- **Nhóm xác định đầy đủ:** mỗi ký tự có đúng một tần suất, nên thuộc đúng một nhóm; các nhóm đôi một rời nhau và phủ hết ký tự có trong `s`.
- **Chọn theo khóa ghép:** "nhiều ký tự nhất" là tiêu chí chính; "tần suất lớn hơn" là tiêu chí phụ khi hòa. So sánh tuple `(kích thước, tần suất)` làm đúng thứ tự ưu tiên này; `max` trả nhóm tốt nhất.
- **Cách 3 (bucket giảm dần):** duyệt `f` từ lớn xuống, chỉ thay `best` khi nhóm mới **lớn hơn hẳn**. Với hai nhóm cùng kích thước, nhóm có `f` lớn hơn được xét trước và không bị thay thế -> đúng luật hòa.
- **Cách 4:** `size[f]` bằng đúng số ký tự có tần suất `f`; chọn `f_best` theo khóa ghép giống Cách 2; lọc ký tự theo `f_best` cho đúng nhóm.
- **Cách 1:** thử mọi `f` có thể (`1..n`), dùng `>=` khi duyệt `f` tăng dần nên khi hòa `f` lớn hơn ghi đè.
- **Thứ tự ký tự:** đề cho phép thứ tự bất kỳ; khi kiểm thử ta so sánh `sorted`.

## 11. Phân tích Time / Space Complexity

`n = len(s) <= 100`, bảng chữ cái `|Σ| = 26`.

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn theo tần suất | `O(n * |Σ| * n)` | `O(|Σ|)` |
| 2. `Counter` + `defaultdict` + `max` | `O(n + |Σ|)` | `O(|Σ|)` |
| 3. Bucket giảm dần | `O(n + |Σ|)` | `O(n + |Σ|)` |
| 4. `Counter` của tần suất | `O(n + |Σ|)` | `O(|Σ|)` |

Cách 2 là cân bằng tốt nhất giữa dễ đọc và hiệu quả.

## 12. Bộ test

Vì thứ tự ký tự trong đáp án tùy ý, ta so sánh `sorted(kết quả)`. Test 1: ví dụ đề và edge case (đều tuân `1 <= len(s) <= 100`, chữ thường).

In [13]:
impls = [SolutionBrute(), SolutionGroup(), SolutionBucket(), SolutionDoubleCount(), Solution()]

def norm(x):
    return "".join(sorted(x))

def oracle(s):
    # độc lập: dùng s.count, duyệt mọi cặp (kích thước, tần suất)
    best_key, best = (-1, -1), ""
    for f in range(1, len(s) + 1):
        grp = sorted({c for c in s if s.count(c) == f})
        if grp and (len(grp), f) > best_key:
            best_key, best = (len(grp), f), "".join(grp)
    return best

tests = [
    ("aaabbbccdddde", "ab"),
    ("abcd", "abcd"),
    ("pfpfgi", "fp"),
    ("a", "a"),
    ("aaaa", "a"),
    ("aabb", "ab"),
    ("abb", "a"),                  # a:1, b:2 -> hai nhóm kích thước 1 -> chọn tần suất lớn: b
    ("abdddd", "ab"),              # nhóm {a,b} kích thước 2 thắng nhóm {d} tần suất 4
    ("aabbbcccc", "c"),            # ba nhóm kích thước 1 -> tần suất lớn nhất: c (4)
    ("z" * 100, "z"),
    ("abcdefghijklmnopqrstuvwxyz", "abcdefghijklmnopqrstuvwxyz"),
]
tests[6] = ("abb", "b")
for s, exp in tests:
    assert 1 <= len(s) <= 100 and s.isalpha() and s.islower()
    assert oracle(s) == norm(exp), (s, oracle(s), exp)
    for im in impls:
        assert norm(im.majorityFrequencyGroup(s)) == norm(exp), (type(im).__name__, s, im.majorityFrequencyGroup(s), exp)
print("tất cả test cố định OK:", len(tests))

tất cả test cố định OK: 11


Test 2: ngẫu nhiên có seed, đúng constraints. Dùng bảng chữ cái nhỏ (nhiều hòa nhóm) và đầy đủ 26 chữ; kiểm tra thêm rằng kết quả đều là các ký tự có **cùng một** tần suất.

In [14]:
import random, string
random.seed(3692)

ties = 0
for t in range(4000):
    n = random.randint(1, 100)
    alpha = random.choice([2, 3, 4, 6, 26])
    s = "".join(random.choice(string.ascii_lowercase[:alpha]) for _ in range(n))
    exp = oracle(s)
    cnt = Counter(s)
    sizes = Counter(cnt.values())
    ties += (list(sizes.values()).count(max(sizes.values())) > 1)
    for im in impls:
        got = im.majorityFrequencyGroup(s)
        assert norm(got) == exp, (type(im).__name__, s, got, exp)
        assert len({cnt[c] for c in got}) == 1
print("4000 test ngẫu nhiên OK; số ca có hòa kích thước =", ties)

4000 test ngẫu nhiên OK; số ca có hòa kích thước = 2191


Test 3: tính chất — hoán vị chuỗi không đổi đáp án; kết quả không rỗng và không có ký tự lặp; kích thước kết quả bằng `max(Counter(cnt.values()).values())`; đổi tên bảng chữ cái (đơn ánh ký tự) không đổi **kích thước** đáp án.

In [15]:
random.seed(692)
for _ in range(1500):
    n = random.randint(1, 100)
    s = "".join(random.choice("abcdef") for _ in range(n))
    res = Solution().majorityFrequencyGroup(s)
    assert res and len(set(res)) == len(res)
    chars = list(s)
    random.shuffle(chars)
    assert norm(Solution().majorityFrequencyGroup("".join(chars))) == norm(res)
    assert len(res) == max(Counter(Counter(s).values()).values())
    shifted = "".join(chr((ord(c) - 97 + 1) % 26 + 97) for c in s)
    assert len(Solution().majorityFrequencyGroup(shifted)) == len(res)
print("tính chất (hoán vị, không rỗng/không lặp, kích thước, đổi tên chữ cái): OK")

tính chất (hoán vị, không rỗng/không lặp, kích thước, đổi tên chữ cái): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: hòa thì chọn tần suất **thấp** hơn

In [16]:
def wrong_tie_low(s):
    cnt = Counter(s)
    groups = defaultdict(list)
    for ch, f in cnt.items():
        groups[f].append(ch)
    best = max(groups, key=lambda f: (len(groups[f]), -f))      # SAI: -f ưu tiên tần suất nhỏ
    return "".join(groups[best])

print("sai :", norm(wrong_tie_low("pfpfgi")))
print("đúng:", norm(Solution().majorityFrequencyGroup("pfpfgi")))
assert norm(wrong_tie_low("pfpfgi")) != norm(Solution().majorityFrequencyGroup("pfpfgi"))

sai : gi
đúng: fp


Vì sao sai: `"pfpfgi"` hòa kích thước `2` giữa nhóm tần suất `2` (`p, f`) và nhóm tần suất `1` (`g, i`); đề chọn tần suất **lớn hơn**, tức `"fp"`, còn cách sai chọn `"gi"`. **Sửa:** khóa `(len, f)`.

### Lỗi 2: chỉ trả **một** ký tự xuất hiện nhiều nhất

In [17]:
def wrong_most_common(s):
    return Counter(s).most_common(1)[0][0]                # SAI: đề trả cả nhóm

print("sai :", wrong_most_common("aaabbbccdddde"))
print("đúng:", norm(Solution().majorityFrequencyGroup("aaabbbccdddde")))
assert wrong_most_common("aaabbbccdddde") != norm(Solution().majorityFrequencyGroup("aaabbbccdddde"))

sai : d
đúng: ab


Vì sao sai: `most_common(1)` trả đúng **một** ký tự (`"d"`, tần suất cao nhất `4`), trong khi đáp án là nhóm `"ab"`. Đề yêu cầu nhóm có **nhiều ký tự nhất**, không phải ký tự **phổ biến nhất**.

### Lỗi 3: chọn nhóm theo **tần suất lớn nhất**, bỏ qua kích thước

In [18]:
def wrong_max_freq_group(s):
    cnt = Counter(s)
    groups = defaultdict(list)
    for ch, f in cnt.items():
        groups[f].append(ch)
    return "".join(groups[max(groups)])                   # SAI: lấy nhóm có f lớn nhất

print("sai :", norm(wrong_max_freq_group("aaabbbccdddde")))
print("đúng:", norm(Solution().majorityFrequencyGroup("aaabbbccdddde")))
assert norm(wrong_max_freq_group("aaabbbccdddde")) != norm(Solution().majorityFrequencyGroup("aaabbbccdddde"))

sai : d
đúng: ab


Vì sao sai: nhóm có tần suất lớn nhất là `{d}` (`4`), kích thước `1`, nhưng nhóm `{a, b}` có kích thước `2`. Tần suất chỉ dùng để **phá hòa**.

### Lỗi 4: đo nhóm bằng **tổng số lần xuất hiện** (`f * kích thước`)

In [19]:
def wrong_weighted(s):
    cnt = Counter(s)
    groups = defaultdict(list)
    for ch, f in cnt.items():
        groups[f].append(ch)
    best = max(groups, key=lambda f: (f * len(groups[f]), f))     # SAI: đề đếm số ký tự khác nhau
    return "".join(groups[best])

print("sai :", norm(wrong_weighted("abdddd")))
print("đúng:", norm(Solution().majorityFrequencyGroup("abdddd")))
assert norm(wrong_weighted("abdddd")) != norm(Solution().majorityFrequencyGroup("abdddd"))

sai : d
đúng: ab


Vì sao sai: nhóm `{a, b}` (tần suất `1`) có `2` ký tự nhưng chỉ `2` lần xuất hiện; nhóm `{d}` có `1` ký tự nhưng `4` lần. Đề đếm **số ký tự khác nhau** trong nhóm nên đáp án đúng `"ab"`, còn cách sai ra `"d"`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Nhóm theo tần suất" -> **đếm hai tầng**: `Counter(s)` rồi gom hoặc `Counter(cnt.values())`.
- "Lớn nhất theo nhiều tiêu chí" -> `max(..., key=lambda x: (tiêu chí 1, tiêu chí 2))`.
- "Hòa thì chọn lớn hơn" -> thêm tiêu chí phụ **dương** vào khóa (`f`, không phải `-f`).
- Bảng chữ cái 26 -> mảng đếm đủ dùng; `Counter` ngắn gọn hơn.
- Đáp án chấp nhận mọi thứ tự -> kiểm thử bằng `sorted`.
- Với bài đếm tần suất, luôn thử các ca: một ký tự, mọi ký tự khác nhau, và hai nhóm hòa kích thước.

## 15. Biến thể của bài

1. Hòa thì chọn nhóm có tần suất **nhỏ** hơn.
2. Trả về **mọi** nhóm có kích thước lớn nhất (không phá hòa).
3. Tìm tần suất xuất hiện **nhiều ký tự khác nhau nhất** (chỉ trả về tần suất).
4. Bài `1207. Unique Number of Occurrences`: các tần suất có đôi một khác nhau không?
5. Bài `1636. Sort Array by Increasing Frequency`: sắp xếp theo tần suất rồi theo giá trị.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn theo tần suất | thử `f = 1..n` | `O(26 n^2)` | `O(26)` | baseline, hiểu đề |
| 2. `Counter` + gom + `max` | khóa `(kích thước, f)` | `O(n)` | `O(26)` | **nên nộp** |
| 3. Bucket giảm dần | `>` chặt tự phá hòa | `O(n)` | `O(n)` | không cần khóa ghép |
| 4. `Counter` của tần suất | đếm kích thước nhóm trước | `O(n)` | `O(26)` | gọn, đúng mô hình hai tầng |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().majorityFrequencyGroup("aabbc")` là gì (sắp xếp ký tự)?

**Bài 2 (tính tay):** `s = "xxyyzzw"`. Tần suất, các nhóm, đáp án?

**Bài 3 (điền TODO):** hoàn thành `group_sizes(s)` trả về dict `{tần suất: số ký tự}`.

**Bài 4 (tự viết lại):** viết `all_max_groups(s)` trả về **list** các nhóm (mỗi nhóm là chuỗi đã sắp) có kích thước lớn nhất, **không** phá hòa.

**Bài 5 (mini challenge):** viết `best_frequency(s)` trả về **tần suất** của nhóm đa số (không cần các ký tự).

In [20]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(norm(Solution().majorityFrequencyGroup("aabbc")))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def group_sizes(s):
    # TODO: Counter(Counter(s).values()) rồi đổi sang dict
    return None

ab


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** đếm ký tự, gom ký tự theo tần suất, đếm số ký tự mỗi nhóm.

**Mức 2 (vừa):** Bài 3: `dict(Counter(Counter(s).values()))`. Bài 4: tìm kích thước lớn nhất rồi liệt kê mọi nhóm có kích thước đó. Bài 5: `max(size, key=lambda f: (size[f], f))`.

**Mức 3 (gần đáp án):** Bài 1: `a=2, b=2, c=1` -> nhóm `{a, b}` kích thước 2 -> `"ab"`. Bài 2: `x=2, y=2, z=2, w=1` -> nhóm `2 -> {x, y, z}` kích thước 3 -> `"xyz"`.

## 19. Đáp án tham khảo cho bài tập

In [21]:
# Bài 1
assert norm(Solution().majorityFrequencyGroup("aabbc")) == "ab"

# Bài 2
assert dict(Counter("xxyyzzw")) == {"x": 2, "y": 2, "z": 2, "w": 1}
assert norm(Solution().majorityFrequencyGroup("xxyyzzw")) == "xyz"

# Bài 3
def group_sizes(s):
    return dict(Counter(Counter(s).values()))
assert group_sizes("pfpfgi") == {2: 2, 1: 2}
assert group_sizes("aaabbbccdddde") == {3: 2, 2: 1, 4: 1, 1: 1}

# Bài 4
def all_max_groups(s):
    cnt = Counter(s)
    groups = defaultdict(list)
    for ch, f in cnt.items():
        groups[f].append(ch)
    top = max(len(g) for g in groups.values())
    return sorted("".join(sorted(g)) for g in groups.values() if len(g) == top)
assert all_max_groups("pfpfgi") == ["fp", "gi"]
assert all_max_groups("aaabbbccdddde") == ["ab"]

# Bài 5
def best_frequency(s):
    size = Counter(Counter(s).values())
    return max(size, key=lambda f: (size[f], f))
assert best_frequency("pfpfgi") == 2
assert best_frequency("aaabbbccdddde") == 3
assert best_frequency("abcd") == 1
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "nhóm theo tần suất rồi chọn nhóm tốt nhất":**

```python
cnt = Counter(s)
groups = defaultdict(list)
for ch, f in cnt.items():
    groups[f].append(ch)
best = max(groups, key=lambda f: (len(groups[f]), f))   # kích thước trước, hòa thì f lớn
return "".join(groups[best])
```

**Checklist trước khi submit:**

- [ ] Đếm tần suất **ký tự**, rồi gom ký tự theo tần suất (hai tầng)?
- [ ] Tiêu chí chính là **số ký tự khác nhau** trong nhóm (không phải tổng số lần)?
- [ ] Hòa kích thước thì chọn **tần suất lớn hơn** (khóa `(len, f)`, không `-f`)?
- [ ] Trả về **cả nhóm** (không chỉ một ký tự phổ biến nhất)?
- [ ] Đã thử `"pfpfgi"` (hòa), `"abcd"` (một nhóm), `"abdddd"` (nhóm nhỏ tần suất thấp thắng)?
- [ ] Chữ ký đúng `majorityFrequencyGroup(self, s: str) -> str`?